# Example Progression 1 — Insurance ChatBot (SDK, interactive)

This notebook is the interactive counterpart of
[`wf_examples/wf_example_progression_1.py`](../../wf_examples/wf_example_progression_1.py).
It takes the **exact same workflow** that the script builds and drives it end-to-end
through the Interactly SDK against the dev server:

1. Build the fully-hydrated workflow config with `build_assistant_workflow()`
   (ported verbatim from the script — only the imports change to `interactly.configs`).
2. Upload it and obtain a live `AsyncWorkflowHandle` with `aupload_and_get_handle()`.
3. Hold a **capped** multi-turn chat (no `input()` — a fixed list of user turns), following
   the interactive pattern from [`02_interactive_workflow.ipynb`](../02_interactive_workflow.ipynb).
4. Fetch the completed run for inspection.
5. **Clean up** — delete the workflow (and with it the run) so no dev resources leak.

The workflow itself is a 3-node Cigna insurance chatbot: a **greeting** node → an
**insurance chatbot** node (self-loops, waiting for the user each turn) → a static
**end-conversation** node reached by a conditional edge when the user says goodbye.

> **These notebooks are async-first.** They use `AsyncWorkflowClient` with top-level `await`,
> which runs directly in Jupyter (the setup cell calls `nest_asyncio.apply()`). Every method
> shown also exists on the synchronous `WorkflowClient` — just drop the `await`.

In [ ]:
# This notebook lives one level below notebooks/, where _bootstrap.py lives. Walk up from the
# working dir to find it, add that dir to sys.path, then import it so `import interactly` works
# with no install needed. (Works whether the kernel cwd is notebooks/ or wf_example_notebooks/.)
import os, sys
from pathlib import Path
import nest_asyncio
from dotenv import load_dotenv

# Required to run asyncio in a Jupyter Notebook
nest_asyncio.apply()

_notebooks_dir = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / '_bootstrap.py').exists():
        _notebooks_dir = _cand
        break
if _notebooks_dir and str(_notebooks_dir) not in sys.path:
    sys.path.insert(0, str(_notebooks_dir))

import _bootstrap  # noqa: F401 - enables `import interactly` (no install needed)

# Find the project root (the workflow_sdk dir that holds .env) and load credentials from it.
project_root = Path.cwd()
while project_root.parent != project_root:
    if (project_root / '.env').exists():
        break
    project_root = project_root.parent
env_path = project_root / '.env'
if env_path.exists():
    load_dotenv(dotenv_path=str(env_path))
    print(f'Loaded .env from: {env_path}')
else:
    print(f'Warning: .env file not found at {env_path}')

In [ ]:
# Interactly credentials are read from environment variables.
#
# Convenience defaults point at the dev "Workflow Illustrations" org; your shell
# environment always wins (setdefault only fills in what you have not set).
os.environ.setdefault('INTERACTLY_BASE_URL', 'https://api-dev.interactly.ai/workflows')
os.environ.setdefault('INTERACTLY_TEAM_ID', '67458e762b7d3dc15aaea5b5')
os.environ.setdefault('INTERACTLY_USER_ID', '687b1a4f745c8e6806c98d91')

# The bearer token is a secret — never hardcode it in the notebook.
# Export it before launching Jupyter:  export INTERACTLY_API_KEY="…"
assert os.environ.get('INTERACTLY_API_KEY'), (
    'Set INTERACTLY_API_KEY in your environment before running this notebook.'
)

print(f"TEAM ID is: {os.getenv('INTERACTLY_TEAM_ID')}")
print(f"USER ID is: {os.getenv('INTERACTLY_USER_ID')}")
print(f"BASE URL is: {os.getenv('INTERACTLY_BASE_URL')}")

In [ ]:
import nest_asyncio

from interactly import AsyncWorkflowClient

# Required to run top-level `await` inside a Jupyter notebook
nest_asyncio.apply()

client = AsyncWorkflowClient()
print('Connected to', client._base_url)

## 1. Build the workflow

Rather than duplicate the workflow definition here, we import `build_assistant_workflow()`
directly from [`wf_examples/wf_example_progression_1.py`](../../wf_examples/wf_example_progression_1.py)
and call it. This guarantees the notebook and the script never drift — the notebook runs the
**exact same** builder. It returns a `WorkflowConfigFullyHydrated` (three nodes + two edges)
and a `dynamic_variables` dict.

In [ ]:
# Import the builder straight from the example script instead of duplicating it, so the
# notebook always runs the exact same workflow definition as wf_examples/wf_example_progression_1.py.
#
# Put wf_examples/ on sys.path so `wf_example_progression_1` — and the `_shared_sdk` bootstrap
# it imports on its first line — resolve. (Importing the module only defines functions; its
# interactive main() is guarded by `if __name__ == "__main__"`, so nothing runs on import.)
_wf_examples = None
for _cand in [Path.cwd(), *Path.cwd().parents]:
    if (_cand / 'wf_examples' / 'wf_example_progression_1.py').exists():
        _wf_examples = _cand / 'wf_examples'
        break
assert _wf_examples, 'Could not locate the wf_examples/ directory.'
if str(_wf_examples) not in sys.path:
    sys.path.insert(0, str(_wf_examples))

from wf_example_progression_1 import build_assistant_workflow

workflow_config_full, dynamic_variables = build_assistant_workflow()
print('Built workflow config:', workflow_config_full.workflow_config.name)
print('Nodes:', [n.name for n in workflow_config_full.node_configs])
print('Edges:', [e.name for e in workflow_config_full.edge_configs])

## 2. Create the workflow on the dev server

Exactly as the script does, we upload the config with `aupload_and_get_handle()`. This
creates the workflow (publishing + activating a `v0` version) and hands back an
`AsyncWorkflowHandle` that tracks the `run_id` across turns for us. We keep the
`workflow_id` so we can delete it during cleanup.

In [ ]:
from interactly import aupload_and_get_handle
from interactly.runtime.handle import AsyncWorkflowHandle

handle: AsyncWorkflowHandle = await aupload_and_get_handle(
    client,
    workflow_config_full,
    dynamic_variables=dynamic_variables,
)
WF_ID = handle.workflow_id
print(f'Uploaded workflow id={WF_ID}')

## 3. Drive a capped multi-turn chat

The script reads user turns from `input()` in an unbounded `while True` loop. A notebook
runs unattended, so instead we send a **fixed, capped list of user turns** (the pattern from
[`02_interactive_workflow.ipynb`](../02_interactive_workflow.ipynb)). The first turn uses
`WorkflowCommand.START`; subsequent turns use `WorkflowCommand.DATA`. The final turn says
goodbye, which satisfies the conditional edge and routes to the static end node — ending the run.

In [ ]:
from langchain_core.messages import HumanMessage

from interactly.configs import (
    LLMNodeRunInput,
    NodesRunInputs,
    WorkflowCommand,
    WorkflowRunInput,
)
from interactly.runtime.events import (
    AssistantResponseEvent,
    BusyWaitForUserMessageEvent,
    EndWorkflowEvent,
)

# A hard cap on events per turn, mirroring the script's infinite-loop guard.
MAX_EVENTS_PER_TURN = 100


async def send_message(user_text: str, *, command: WorkflowCommand = WorkflowCommand.DATA):
    """Send one user turn on thread \"0\", print assistant bubbles, return the raw events."""
    print(f'User: {user_text}')
    run_input = WorkflowRunInput(
        command=command,
        thread_to_node_inputs={
            '0': NodesRunInputs(
                node_run_inputs=[LLMNodeRunInput(messages=[HumanMessage(content=user_text)])]
            )
        },
        dynamic_variables=dynamic_variables,
    )
    events = []
    async for event in handle.arun(run_input):
        events.append(event)
        if isinstance(event, AssistantResponseEvent) and event.content:
            print(f'  Assistant ({event.origin_node_name}): {event.content}')
        elif isinstance(event, BusyWaitForUserMessageEvent):
            print(f'  (waiting for user at {event.origin_node_name})')
        elif isinstance(event, EndWorkflowEvent):
            print('  (workflow ended)')
        if len(events) > MAX_EVENTS_PER_TURN:
            print('  ! stopping turn early (event cap reached)')
            break
    return events

In [ ]:
# A fixed script of user turns (capped). The last turn says goodbye to end the workflow.
USER_TURNS = [
    'Hi there!',
    'Can you explain what a copay is?',
    "How do I find a doctor who's in my network?",
    "That's all I needed. Thank you, goodbye!",
]

ended = False
for i, text in enumerate(USER_TURNS):
    print('=' * 80)
    print(f'--- Turn {i + 1}/{len(USER_TURNS)} ---')
    cmd = WorkflowCommand.START if i == 0 else WorkflowCommand.DATA
    turn_events = await send_message(text, command=cmd)
    if any(isinstance(e, EndWorkflowEvent) for e in turn_events):
        ended = True
        print('\nWorkflow reached its end node — stopping the chat.')
        break

print(f'\nSession run_id: {handle.run_id}')
print(f'Workflow ended cleanly: {ended}')
RUN_ID = handle.run_id

## 4. Fetch the completed run

The handle tracked the `run_id` for us. We fetch the run back and print each
input/output pair the server persisted for this conversation.

In [ ]:
from interactly import Run
from interactly.runtime.events import parse_event

run: Run = await client.runs.get(RUN_ID)
print(f'Run {run.id}  status={run.status}  started={run.started_at}')


def _get(obj, key):
    return getattr(obj, key, None) if not isinstance(obj, dict) else obj.get(key)


for i, pair in enumerate(run.input_output_pairs):
    run_output = _get(pair, 'run_output')
    events = _get(run_output, 'events') or []
    assistant_texts = []
    for raw in events:
        try:
            ev = parse_event(raw)
        except Exception:
            continue
        if isinstance(ev, AssistantResponseEvent) and ev.content:
            assistant_texts.append(ev.content)
    print(f'  Pair {i}: {len(events)} event(s); assistant said: {assistant_texts}')

## 5. Cleanup

Delete the workflow we created. Deleting the workflow removes its versions and the run
created above, so no resources are left behind on the dev server.

In [ ]:
await client.workflows.delete(WF_ID)
print(f'Workflow {WF_ID} deleted.')

await client.close()
print('Client closed.')